<a href="https://colab.research.google.com/github/joryhh/Capstone-project-agentic-AI-systems-engineering/blob/main/agents/agent1_and_graph_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install tavily-python -q

In [15]:
!pip show tavily-python

Name: tavily-python
Version: 0.7.27
Summary: Python wrapper for the Tavily API
Home-page: https://github.com/tavily-ai/tavily-python
Author: Tavily AI
Author-email: support@tavily.com
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: httpx, requests, tiktoken
Required-by: 


In [17]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [13]:
# ============================================================
# STEP 1 — SHARED STATE SCHEMA
# ============================================================

from typing import TypedDict, List, Dict, Optional, Literal, Annotated
import operator

# ---- Sub-schemas: the actual inter-agent "message contract" ----

class SearchResult(TypedDict):
    source: str            # "flight_search" | "hotel_search" | "attraction_search"
    query: str
    raw_content: str
    url: Optional[str]

class ReActStep(TypedDict):
    thought: str
    action: str
    action_input: str
    observation: str

class ItineraryItem(TypedDict):
    day: int
    activity: str
    location: str
    estimated_cost: float
    category: Literal["flight", "accommodation", "transit", "activity", "food", "entry_fee"]

class BudgetBreakdown(TypedDict):
    total_estimated_cost: float
    budget_limit: float
    over_budget: bool
    over_budget_amount: float
    cost_by_category: Dict[str, float]
    constraint_for_replanning: Optional[str]  # <- the concrete constraint B sends back to Agent 1

class GuardrailLog(TypedDict):
    check_type: Literal["prompt_injection", "pii_masking"]
    triggered: bool
    details: str
    timestamp: str

# ---- The shared State object ----

class TravelState(TypedDict):
    # Input (set once, at invoke time)
    user_request: str
    destination: str
    travel_dates: str
    budget_limit: float
    traveler_preferences: List[str]

    # Agent 1 (Planner / ReAct)
    react_trace: Annotated[List[ReActStep], operator.add]
    search_results: Annotated[List[SearchResult], operator.add]
    draft_itinerary: List[ItineraryItem]

    # Agent 2 (Budget)
    budget_analysis: Optional[BudgetBreakdown]

    # Agent 3 (Audit)
    final_summary: Optional[str]
    audit_notes: List[str]

    # Guardrails
    guardrail_logs: Annotated[List[GuardrailLog], operator.add]
    pii_masked: bool

    # Control flow
    iteration_count: int
    max_iterations: int
    replan_reason: Optional[str]

    # HITL
    human_approved: Optional[bool]
    human_feedback: Optional[str]

    # Cross-cutting
    execution_logs: Annotated[List[str], operator.add]
    status: Literal["in_progress", "awaiting_human", "completed", "failed"]

In [4]:
# CORRECT — partial return, reducer appends automatically
def agent1_planner_node(state: TravelState) -> dict:
    return {
        "execution_logs": ["Agent 1 (Planner) stub executed"],
        "iteration_count": state["iteration_count"] + 1,
    }




In [5]:
# ============================================================
# STEP 4 — GRAPH SKELETON (stub nodes, real edges + real conditional logic)
# ============================================================

from langgraph.graph import StateGraph, END
from typing import Literal

# ---- Stub nodes  ----

def agent1_planner_node(state: TravelState) -> dict:
    print("  [Agent 1 - Planner] STUB running.")
    return {
        "execution_logs": ["Agent 1 (Planner) stub executed"],
        "iteration_count": state["iteration_count"] + 1,
    }

def agent2_budget_node(state: TravelState) -> dict:
    print("  [Agent 2 - Budget] STUB running.")
    return {
        "execution_logs": ["Agent 2 (Budget) stub executed"],
    }

def agent3_audit_node(state: TravelState) -> dict:
    print("  [Agent 3 - Audit] STUB running.")
    return {
        "execution_logs": ["Agent 3 (Audit) stub executed"],
        "status": "completed",
    }

def human_review_node(state: TravelState) -> dict:
    print("  [Human Review] STUB — B will build this as the real HITL interrupt.")
    return {
        "execution_logs": ["Human review stub executed"],
    }

# ---- The router: your termination guard + conditional edge ----

def budget_router(state: TravelState) -> Literal["replan", "proceed"]:
    # Termination guard FIRST — this must win regardless of budget status,
    # or an unfixable budget constraint loops forever.
    if state["iteration_count"] >= state["max_iterations"]:
        print(f"  [Router] Max iterations ({state['max_iterations']}) reached -> forcing proceed.")
        return "proceed"

    budget = state.get("budget_analysis")
    if budget and budget["over_budget"]:
        print(f"  [Router] Over budget by {budget['over_budget_amount']} -> replanning.")
        return "replan"

    print("  [Router] Within budget -> proceeding.")
    return "proceed"

# ---- Assemble the graph ----

workflow = StateGraph(TravelState)

workflow.add_node("agent1_planner", agent1_planner_node)
workflow.add_node("agent2_budget", agent2_budget_node)
workflow.add_node("agent3_audit", agent3_audit_node)
workflow.add_node("human_review", human_review_node)

workflow.set_entry_point("agent1_planner")
workflow.add_edge("agent1_planner", "agent2_budget")

workflow.add_conditional_edges(
    "agent2_budget",
    budget_router,
    {
        "replan": "agent1_planner",   # loop back
        "proceed": "agent3_audit",    # move forward
    },
)

workflow.add_edge("agent3_audit", "human_review")
workflow.add_edge("human_review", END)

app = workflow.compile()
print("Graph compiled and ready.")

Graph compiled and ready.


In [6]:
# ============================================================
# STEP 5 — SMOKE TEST: run the skeleton, confirm the loop + guard work
# ============================================================

initial_state: TravelState = {
    "user_request": "5-day trip to Kyoto, mid-range budget",
    "destination": "Kyoto",
    "travel_dates": "2026-11-10 to 2026-11-15",
    "budget_limit": 2000.0,
    "traveler_preferences": ["food", "temples"],
    "react_trace": [],
    "search_results": [],
    "draft_itinerary": [],
    "budget_analysis": None,
    "final_summary": None,
    "audit_notes": [],
    "guardrail_logs": [],
    "pii_masked": False,
    "iteration_count": 0,
    "max_iterations": 3,
    "replan_reason": None,
    "human_approved": None,
    "human_feedback": None,
    "execution_logs": [],
    "status": "in_progress",
}

# Belt-and-suspenders: LangGraph's own recursion_limit as a second safety net,
# independent of your max_iterations field — good practice for any graph with a cycle.
final_state = app.invoke(initial_state, config={"recursion_limit": 50})

print("\nFinal status:", final_state["status"])
print("Iterations used:", final_state["iteration_count"])
for log in final_state["execution_logs"]:
    print(log)

  [Agent 1 - Planner] STUB running.
  [Agent 2 - Budget] STUB running.
  [Router] Within budget -> proceeding.
  [Agent 3 - Audit] STUB running.
  [Human Review] STUB — B will build this as the real HITL interrupt.

Final status: completed
Iterations used: 1
Agent 1 (Planner) stub executed
Agent 2 (Budget) stub executed
Agent 3 (Audit) stub executed
Human review stub executed


In [18]:
 # ============================================================
# AGENT 1 — DESTINATION PLANNER (real ReAct: Thought -> Action -> Observation)
# ============================================================
# pip install langchain-groq langchain-core tavily-python

import os
import json
from datetime import datetime, timezone
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_groq import ChatGroq
from tavily import TavilyClient

# ---- Keys: pull from environment, NEVER hardcode ----
# In Colab: use the "Secrets" tab (key icon on the left) to store these,
# then: from google.colab import userdata; os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")

if not GROQ_API_KEY or not TAVILY_API_KEY:
    raise EnvironmentError(
        "Missing GROQ_API_KEY or TAVILY_API_KEY in environment. "
        "Set them via Colab secrets or a local .env — do not hardcode."
    )

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

# ---- The real tool Agent 1 calls ----

def search_travel_info(query: str, max_retries: int = 3) -> str:
    """Search the web for real, current travel information: destinations,
    activities, attractions, seasonal notes, or logistics.
    Retries on transient network failures with exponential backoff —
    this is the retry/fallback path the rubric asks to demonstrate."""
    import time

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            results = tavily_client.search(query=query, max_results=3, search_depth="basic")
            formatted = []
            for r in results.get("results", []):
                formatted.append(f"- {r['title']}: {r['content'][:300]} (source: {r['url']})")
            return "\n".join(formatted) if formatted else "No results found."
        except Exception as e:
            last_error = e
            wait = 2 ** attempt  # 2s, 4s, 8s
            print(f"  [search_travel_info] Attempt {attempt}/{max_retries} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    # All retries exhausted -> fallback: don't crash the whole graph, degrade gracefully
    print(f"  [search_travel_info] All {max_retries} attempts failed. Falling back to empty result.")
    return f"[SEARCH UNAVAILABLE after {max_retries} retries: {last_error}]"

# Tool schema for Groq function calling
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "search_travel_info",
            "description": "Search the web for real, current travel information about a destination: attractions, activities, neighborhoods, seasonal considerations, or logistics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query, e.g. 'best temples to visit in Kyoto November'",
                    }
                },
                "required": ["query"],
            },
        },
    }
]

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_with_tools = llm.bind_tools(TOOLS_SCHEMA)

MAX_REACT_STEPS = 4  # hard cap on the inner ReAct loop, independent of the graph's max_iterations


def agent1_planner_node(state: dict) -> dict:
    """
    Real ReAct loop: the model reasons (Thought), decides to call the search
    tool (Action), the tool actually runs (Observation), and the loop repeats
    until the model has enough information to produce a final itinerary.
    """
    print("  [Agent 1 - Planner] Starting ReAct loop...")

    destination = state.get("destination", "")
    dates = state.get("travel_dates", "")
    prefs = ", ".join(state.get("traveler_preferences", []))
    budget = state.get("budget_limit", "unspecified")
    replan_reason = state.get("replan_reason")  # set by Agent 2 on loop-back

    system_prompt = f"""You are a travel destination planning agent using the ReAct pattern.
For each turn, first reason step by step (your Thought), then decide whether to call
the search_travel_info tool to get real information, or if you have enough information,
produce a final structured day-by-day itinerary.

Destination: {destination}
Dates: {dates}
Preferences: {prefs}
Budget limit: {budget}
{"IMPORTANT constraint from budget review: " + replan_reason if replan_reason else ""}

When you are ready to finalize, respond with plain text starting with 'FINAL ITINERARY:'
followed by a day-by-day breakdown with estimated costs per item."""

    messages = [SystemMessage(content=system_prompt),
                HumanMessage(content="Begin planning. Think first, then act.")]

    react_trace = []
    search_results = []
    step = 0

    while step < MAX_REACT_STEPS:
        step += 1
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        tool_calls = getattr(response, "tool_calls", None)

        if not tool_calls:
            # Model decided it has enough info -> this is the final Thought, no Action
            react_trace.append({
                "thought": response.content[:500],
                "action": "none (final answer)",
                "action_input": "",
                "observation": "",
            })
            print(f"  [Agent 1] Step {step}: model finalized without further tool calls.")
            break

        for call in tool_calls:
            query = call["args"].get("query", "")
            print(f"  [Agent 1] Step {step}: Action=search_travel_info | query='{query}'")

            observation = search_travel_info(query)

            react_trace.append({
                "thought": (response.content or "(reasoning to call tool)")[:500],
                "action": "search_travel_info",
                "action_input": query,
                "observation": observation[:500],
            })
            search_results.append({
                "source": "attraction_search",
                "query": query,
                "raw_content": observation,
                "url": None,
            })

            messages.append(ToolMessage(content=observation, tool_call_id=call["id"]))

    else:
        # Loop hit MAX_REACT_STEPS without a natural finalization
        print(f"  [Agent 1] Hit MAX_REACT_STEPS ({MAX_REACT_STEPS}) — forcing final response.")
        response = llm.invoke(messages + [HumanMessage(content="Finalize now with FINAL ITINERARY: based on what you have.")])
        messages.append(response)

    final_text = messages[-1].content if isinstance(messages[-1], AIMessage) else response.content

    print(f"  [Agent 1] ReAct loop complete after {step} step(s), {len(search_results)} tool call(s).")

    return {
        "react_trace": react_trace,
        "search_results": search_results,
        "draft_itinerary": [],  # parse final_text into structured ItineraryItem list here if needed
        "execution_logs": [
            f"Agent 1 (Planner) completed ReAct loop: {step} step(s), "
            f"{len(search_results)} real tool call(s), at {datetime.now(timezone.utc).isoformat()}"
        ],
        "iteration_count": state["iteration_count"] + 1,
        "_raw_plan_text": final_text,  # optional: drop if not in your TravelState schema
    }


# ---- Standalone smoke test ----
if __name__ == "__main__":
    mock_state = {
        "destination": "Kyoto",
        "travel_dates": "2026-11-10 to 2026-11-15",
        "traveler_preferences": ["food", "temples"],
        "budget_limit": 2000.0,
        "replan_reason": None,
        "iteration_count": 0,
    }
    result = agent1_planner_node(mock_state)
    print("\n--- react_trace ---")
    for step in result["react_trace"]:
        print(json.dumps(step, indent=2, ensure_ascii=False))
    print("\n--- final plan text ---")
    print(result.get("_raw_plan_text"))



  [Agent 1 - Planner] Starting ReAct loop...
  [Agent 1] Step 1: Action=search_travel_info | query='Kyoto temples and food November 2026'
  [Agent 1] Step 1: Action=search_travel_info | query='best food in Kyoto November 2026'
  [Agent 1] Step 1: Action=search_travel_info | query='Kyoto travel logistics November 2026'
  [Agent 1] Step 2: model finalized without further tool calls.
  [Agent 1] ReAct loop complete after 2 step(s), 3 tool call(s).

--- react_trace ---
{
  "thought": "To plan a trip to Kyoto from 2026-11-10 to 2026-11-15, considering the preferences for food and temples, and a budget limit of 2000.0, I need to break down the process into manageable steps. \n\nFirst, I should identify the most famous and relevant temples in Kyoto that align with the travelers' interests. Kyoto is known for its numerous temples, each with its own unique history and architectural style. Some of the most popular temples include Kinkaku-ji (Golden Pavilion), Fushimi Inari Shrine, an",
  "action

In [19]:
mock_state = {
    "destination": "Kyoto",
    "travel_dates": "2026-11-10 to 2026-11-15",
    "traveler_preferences": ["food", "temples"],
    "budget_limit": 2000.0,
    "replan_reason": None,
    "iteration_count": 0,
}
result = agent1_planner_node(mock_state)

print("\n--- react_trace ---")
for step in result["react_trace"]:
    print(json.dumps(step, indent=2, ensure_ascii=False))

print("\n--- final plan text ---")
print(result.get("_raw_plan_text"))

  [Agent 1 - Planner] Starting ReAct loop...
  [Agent 1] Step 1: Action=search_travel_info | query='best temples to visit in Kyoto November'
  [Agent 1] Step 1: Action=search_travel_info | query='traditional food experiences in Kyoto'
  [Agent 1] Step 2: Action=search_travel_info | query='Kyoto budget food options'
  [Agent 1] Step 2: Action=search_travel_info | query='Kyoto temple entrance fees'
  [Agent 1] Step 2: Action=search_travel_info | query='best time to visit Kyoto temples for fall colors'
  [Agent 1] Step 3: model finalized without further tool calls.
  [Agent 1] ReAct loop complete after 3 step(s), 5 tool call(s).

--- react_trace ---
{
  "thought": "To plan a trip to Kyoto from 2026-11-10 to 2026-11-15, considering the preferences for food and temples, and a budget limit of 2000.0, I need to break down the process into manageable steps. \n\nFirst, I should identify the most famous and relevant temples in Kyoto that align with the travelers' interests. Kyoto is known for it

In [20]:
# ============================================================
# LOOP TEST — proves the replan loop (D2) actually fires
# ============================================================


from langgraph.graph import StateGraph, END
from typing import Literal

# ---- Mock Agent 2: forces over_budget for the first 2 iterations, then resolves ----
# This is ONLY for proving the loop mechanics. Member B will replace this with the
# real cost-estimation logic — but the router, the loop-back edge, and the guard
# are exactly what will run against the real Agent 2 too.

def mock_agent2_budget_node(state: dict) -> dict:
    print(f"  [Agent 2 - Budget MOCK] Checking budget at iteration {state['iteration_count']}...")

    if state["iteration_count"] < 3:
        # Force an over-budget result so we can prove the loop-back actually happens
        analysis = {
            "total_estimated_cost": 2600.0,
            "budget_limit": state.get("budget_limit", 2000.0),
            "over_budget": True,
            "over_budget_amount": 600.0,
            "cost_by_category": {"flight": 900.0, "accommodation": 1200.0, "activity": 500.0},
            "constraint_for_replanning": "Cut estimated cost by at least $600 — prioritize free temples/shrines and skip the guided Fushimi Inari tour.",
        }
        print(f"  [Agent 2 - Budget MOCK] OVER budget by {analysis['over_budget_amount']} -> will trigger replan.")
    else:
        analysis = {
            "total_estimated_cost": 1850.0,
            "budget_limit": state.get("budget_limit", 2000.0),
            "over_budget": False,
            "over_budget_amount": 0.0,
            "cost_by_category": {"flight": 900.0, "accommodation": 700.0, "activity": 250.0},
            "constraint_for_replanning": None,
        }
        print(f"  [Agent 2 - Budget MOCK] Within budget -> will proceed.")

    return {
        "budget_analysis": analysis,
        "replan_reason": analysis["constraint_for_replanning"],
        "execution_logs": [f"Agent 2 (Budget MOCK) evaluated at iteration {state['iteration_count']}"],
    }


def agent3_audit_node(state: dict) -> dict:
    print("  [Agent 3 - Audit] STUB running (Member C will build the real version).")
    return {
        "execution_logs": ["Agent 3 (Audit) stub executed"],
        "status": "completed",
    }


def human_review_node(state: dict) -> dict:
    print("  [Human Review] STUB (Member B will build the real HITL interrupt).")
    return {
        "execution_logs": ["Human review stub executed"],
    }


def budget_router(state: dict) -> Literal["replan", "proceed"]:
    if state["iteration_count"] >= state["max_iterations"]:
        print(f"  [Router] Max iterations ({state['max_iterations']}) reached -> forcing proceed.")
        return "proceed"

    budget = state.get("budget_analysis")
    if budget and budget["over_budget"]:
        print(f"  [Router] Over budget by {budget['over_budget_amount']} -> replanning.")
        return "replan"

    print("  [Router] Within budget -> proceeding.")
    return "proceed"


# ---- Build the graph with the REAL Agent 1 + mock Agent 2 ----

test_workflow = StateGraph(TravelState)

test_workflow.add_node("agent1_planner", agent1_planner_node)       # the real one, from before
test_workflow.add_node("agent2_budget", mock_agent2_budget_node)     # mock, forces the loop
test_workflow.add_node("agent3_audit", agent3_audit_node)
test_workflow.add_node("human_review", human_review_node)

test_workflow.set_entry_point("agent1_planner")
test_workflow.add_edge("agent1_planner", "agent2_budget")

test_workflow.add_conditional_edges(
    "agent2_budget",
    budget_router,
    {
        "replan": "agent1_planner",
        "proceed": "agent3_audit",
    },
)

test_workflow.add_edge("agent3_audit", "human_review")
test_workflow.add_edge("human_review", END)

test_app = test_workflow.compile()
print("Test graph compiled.\n")

# ---- Run it ----

test_initial_state = {
    "destination": "Kyoto","travel_dates": "2026-11-10 to 2026-11-15",
    "budget_limit": 2000.0,
    "traveler_preferences": ["food", "temples"],
    "react_trace": [],
    "search_results": [],
    "draft_itinerary": [],
    "budget_analysis": None,
    "final_summary": None,
    "audit_notes": [],
    "guardrail_logs": [],
    "pii_masked": False,
    "iteration_count": 0,
    "max_iterations": 3,
    "replan_reason": None,
    "human_approved": None,
    "human_feedback": None,
    "execution_logs": [],
    "status": "in_progress",
}

print("=" * 60)
print("RUNNING FULL GRAPH — expect: replan, replan, then proceed")
print("=" * 60)

final_state = test_app.invoke(test_initial_state, config={"recursion_limit": 50})

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)
print("Final status:", final_state["status"])
print("Iterations used:", final_state["iteration_count"])
print("Total Agent 1 tool calls across all iterations:", len(final_state["search_results"]))
print("\nExecution log (proves the loop fired):")
for log in final_state["execution_logs"]:
    print(" -", log)

Test graph compiled.

RUNNING FULL GRAPH — expect: replan, replan, then proceed
  [Agent 1 - Planner] Starting ReAct loop...
  [Agent 1] Step 1: Action=search_travel_info | query='Kyoto travel November 2026 temples food logistics'
  [Agent 1] Step 2: Action=search_travel_info | query='Kyoto temple entrance fees and food costs November 2026'
  [Agent 1] Step 3: Action=search_travel_info | query='best food in Kyoto November 2026'
  [Agent 1] Step 4: model finalized without further tool calls.
  [Agent 1] ReAct loop complete after 4 step(s), 3 tool call(s).
  [Agent 2 - Budget MOCK] Checking budget at iteration 1...
  [Agent 2 - Budget MOCK] OVER budget by 600.0 -> will trigger replan.
  [Router] Over budget by 600.0 -> replanning.
  [Agent 1 - Planner] Starting ReAct loop...
  [Agent 1] Step 1: Action=search_travel_info | query='free temples and shrines in Kyoto'
  [Agent 1] Step 1: Action=search_travel_info | query='affordable food in Kyoto'
  [Agent 1] Step 1: Action=search_travel_info